<a href="https://colab.research.google.com/github/sukhadaghewari/wacv2027_ri_benchmark/blob/main/notebooks/09_score_ri_predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RI evaluation: thresholds on the development set, scores on the test set

Every model answers each case with a probability, `ri_probability`, and never sees a threshold. This notebook turns those answers into the paper's numbers, in two strictly separate steps:

1. **Development set (tuning).** For each model, try thresholds and keep the one with the best Peirce skill score (PSS). Its value is saved to `ri_dev<VERSION>_best_thresholds.csv` and then **frozen**.
2. **Test set (evaluation, no tuning).** Score the probabilities directly (ROC-AUC, PR-AUC, Brier skill score), and score yes/no calls made with each model's frozen threshold. 95% confidence intervals come from a bootstrap over storms.

Then the checks: paired model comparisons, scores by phase, calibration and the history-gap robustness check.

Inputs are the results files `results/ri_prediction/dev/<provider>/<model>_dev_2016_20ri_20nori<VERSION>.jsonl.gz` (development) and `results/ri_prediction/test/<provider>/<model>_test_2017-2025_50ri_50nori<VERSION>.jsonl.gz` (test), with `<provider>` = `claude` or `openai`, where `<VERSION>` is the setting: `_text_only` or `_text_image`. The trained baselines from notebook 10 are read from `results/ri_prediction/<dev or test>/ml/<model>_<case file name>.csv` (`case_id`, `ri_probability`); they exist for `_text_only` only. A model is included when both files exist. All tables are written to `results/analysis/`.

## 1. Setup

Works out the repo folder and makes `src/` importable, as in the other notebooks. In Colab it first clones this repo.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "wacv2027_ri_benchmark"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

## 2. Settings

- **Seeds:** fixed, so every number is reproduced exactly on every run.
- **Bootstrap sizes:** 1,000 resamples for the development-set threshold-stability check, and 2,000 for the test-set confidence intervals.
- **`BEST_MODEL`:** the model the others are compared with in the paired comparisons.
- **Climatology:** the reference forecast for the Brier skill score. It is always the development-set RI rate (94/720), a value known before the test run.
- **`VERSION`:** which setting to evaluate: tabular only (`"_text_only"`) or tabular + pictures (`"_text_image"`). Each setting gets its own thresholds and tables (for example `ri_dev_text_image_best_thresholds.csv`); the cases and the bootstrap are the same, so versions can be compared directly.

In [ ]:
import json

import numpy as np
import pandas as pd
from prompts import parse_response, read_answers, read_records

VERSION = os.environ.get("RI_VERSION", "_text_only")   # setting: "_text_only" (tabular) or "_text_image" (tabular + pictures)
DEV_CASES = ROOT / "model_inputs" / "ri_prediction" / "dev" / f"dev_2016_20ri_20nori{VERSION}.jsonl.gz"
TEST_CASES = ROOT / "model_inputs" / "ri_prediction" / "test" / f"test_2017-2025_50ri_50nori{VERSION}.jsonl.gz"
DEV_RESULTS = ROOT / "results" / "ri_prediction" / "dev"
TEST_RESULTS = ROOT / "results" / "ri_prediction" / "test"
DATA_CSV = ROOT / "data" / "ibtracs_era5_sat_2016_2025.csv"
ANALYSIS_DIR = ROOT / "results" / "analysis"
for s in ["dev", "test"]:
    (ANALYSIS_DIR / s).mkdir(parents=True, exist_ok=True)

BEST_MODEL = "claude-sonnet-5-5"   # chosen on the development set (best dev ROC-AUC, PR-AUC and BSS)
DEV_SEED, DEV_BOOTSTRAP = 0, 1000
TEST_SEED, TEST_BOOTSTRAP = 42, 2000
CALIBRATION_BINS = [0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.01]

pd.set_option("display.width", 250, "display.max_columns", 30)

## 3. Read the answers

For each split, the truth comes from the case file and the answers from each model's results file. Every results file must answer exactly the cases in its case file. An answer that can't be parsed is reported, and it is left out of that model's scores.

In [ ]:
def stem(path):
    return path.name.removesuffix(".jsonl.gz")


def load_split(case_file, results_dir):
    truth = pd.DataFrame(read_records(case_file)).drop(columns="prompt")
    probs = {}
    for path in sorted(results_dir.glob(f"*/*_{stem(case_file)}.jsonl.gz")):    # claude/, openai/
        model = path.name.removesuffix(f"_{stem(case_file)}.jsonl.gz")
        answers = read_answers(path, parse_response)
        assert set(answers["case_id"]) == set(truth["case_id"]), f"{path.name} does not answer exactly the cases in {case_file.name}"
        bad = answers[~answers["parse_ok"]]
        if len(bad):
            print(f"{path.name}: {len(bad)} unparseable answers, left out: {bad['case_id'].tolist()[:5]}")
        probs[model] = truth["case_id"].map(answers.set_index("case_id")["ri_probability"]).astype(float).values
    for path in sorted(results_dir.glob(f"ml/*_{stem(case_file)}.csv")):          # trained baselines (notebook 10)
        model = path.name.removesuffix(f"_{stem(case_file)}.csv")
        answers = pd.read_csv(path)
        assert set(answers["case_id"]) == set(truth["case_id"]), f"{path.name} does not answer exactly the cases in {case_file.name}"
        probs[model] = truth["case_id"].map(answers.set_index("case_id")["ri_probability"]).astype(float).values
    return truth, probs


dev, dev_p = load_split(DEV_CASES, DEV_RESULTS)
test, test_p = load_split(TEST_CASES, TEST_RESULTS)
models = [m for m in dev_p if m in test_p]
print("models:", models)
print(f"development: {len(dev)} cases, {int(dev['ri_tplus24'].sum())} RI | test: {len(test)} cases, {int(test['ri_tplus24'].sum())} RI")

CLIM = dev["ri_tplus24"].mean()       # Brier reference: the development-set RI rate, fixed before the test run
print(f"climatology (development RI rate): {CLIM:.4f}")

## 4. Scores

- **ROC-AUC:** how well the probabilities rank RI cases above non-RI cases. 0.5 is chance.
- **PR-AUC (average precision):** precision averaged over recall. Chance equals the RI rate.
- **Brier skill score (BSS):** improvement in mean squared error over always forecasting the climatology. 0 means no better than climatology.
- **Yes/no calls at a threshold** (`p >= threshold` means yes):
  - **POD:** the share of RI cases called yes.
  - **FAR:** the share of yes calls that were wrong.
  - **POFD:** the share of non-RI cases called yes.
  - **PSS** = POD − POFD.
  - **CSI** = hits / (hits + misses + false alarms).
  - **F1.**

In [ ]:
def roc_auc(y, p):
    """ROC-AUC from ranks (tied probabilities share their average rank)."""
    r = pd.Series(p).rank().values
    n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


def pr_auc(y, p):
    """Average precision: precision at each distinct probability, weighted by the recall gained there."""
    df = pd.DataFrame({"y": y, "p": p}).groupby("p")["y"].agg(["sum", "size"]).sort_index(ascending=False)
    tp = df["sum"].cumsum(); predicted = df["size"].cumsum()
    return float(np.sum(np.diff(np.r_[0, (tp / y.sum()).values]) * (tp / predicted).values))


def bss(y, p, weights=None):
    w = np.ones(len(y)) if weights is None else weights
    return 1 - np.average((p - y) ** 2, weights=w) / np.average((CLIM - y) ** 2, weights=w)


def at_threshold(y, p, t):
    yes = p >= t
    tp = (yes & (y == 1)).sum(); fp = (yes & (y == 0)).sum(); fn = (~yes & (y == 1)).sum(); tn = (~yes & (y == 0)).sum()
    pod = tp / (tp + fn); pofd = fp / (fp + tn)
    return {"threshold": t, "predicted_yes": int(yes.sum()), "POD": pod, "FAR": fp / (tp + fp) if tp + fp else np.nan,
            "POFD": pofd, "PSS": pod - pofd, "CSI": tp / (tp + fp + fn), "F1": 2 * tp / (2 * tp + fp + fn)}


def storm_bootstrap(truth, rng, n):
    """n resamples of the cases, drawing whole storms with replacement, separately from RI and non-RI storms."""
    sid = truth["sid"].values; y = truth["ri_tplus24"].values
    storms = pd.unique(sid); idx = {s: np.where(sid == s)[0] for s in storms}
    ri = [s for s in storms if y[idx[s]].any()]; no = [s for s in storms if not y[idx[s]].any()]
    for _ in range(n):
        yield np.concatenate([idx[s] for s in rng.choice(ri, len(ri))] + [idx[s] for s in rng.choice(no, len(no))])

## 5. Development set: choose and freeze each model's threshold

- **Thresholds tried:** every distinct probability the model gave, plus a 0.05 grid.
- **Choice:** the threshold with the highest PSS is kept. The best-F1 threshold is shown for reference.
- **Stability:** the development storms are resampled 1,000 times (the same resamples for every model), and each time the best threshold is found again. The 90% range shows how stable the choice is.
- **Output:** the chosen thresholds are saved, and section 6 reads them back from the file.

In [ ]:
y_dev = dev["ri_tplus24"].values.astype(int)
quality, sweeps, best = [], [], []
for m in models:
    rng = np.random.default_rng(DEV_SEED)          # same resamples for every model, whatever models are present
    ok = ~np.isnan(dev_p[m]); y = y_dev[ok]; p = dev_p[m][ok]; truth = dev[ok].reset_index(drop=True)
    base = y.mean(); brier = np.mean((p - y) ** 2)
    quality.append({"model": m, "cases": len(y), "RI cases": int(y.sum()), "distinct probabilities": len(np.unique(p)),
                    "mean predicted p": round(p.mean(), 3), "observed RI rate": round(base, 3),
                    "ROC-AUC": round(roc_auc(y, p), 3), "PR-AUC": round(pr_auc(y, p), 3), "random PR-AUC": round(base, 3),
                    "Brier": round(brier, 4), "BSS vs dev climatology": round(1 - brier / np.mean((base - y) ** 2), 3)})

    candidates = np.unique(np.r_[np.round(np.arange(0.05, 0.96, 0.05), 2), p])
    sweep = pd.DataFrame([at_threshold(y, p, t) for t in candidates]); sweep["model"] = m
    sweeps.append(sweep)
    b_pss = sweep.loc[sweep["PSS"].idxmax()]; b_f1 = sweep.loc[sweep["F1"].idxmax()]

    picks = []
    for take in storm_bootstrap(truth, rng, DEV_BOOTSTRAP):
        if y[take].sum() == 0:
            continue
        picks.append(candidates[int(np.argmax([at_threshold(y[take], p[take], t)["PSS"] for t in candidates]))])
    best.append({"model": m, "best threshold (PSS)": b_pss["threshold"], "PSS": round(b_pss["PSS"], 3), "POD": round(b_pss["POD"], 3),
                 "FAR": round(b_pss["FAR"], 3), "CSI": round(b_pss["CSI"], 3),
                 "best threshold (F1)": b_f1["threshold"], "F1": round(b_f1["F1"], 3), "POD@F1": round(b_f1["POD"], 3), "FAR@F1": round(b_f1["FAR"], 3),
                 "PSS-best threshold, 90% bootstrap range": f"{np.percentile(picks, 5):.2f}-{np.percentile(picks, 95):.2f}"})

pd.DataFrame(quality).to_csv(ANALYSIS_DIR / "dev" / f"ri_dev{VERSION}_threshold_free.csv", index=False)
pd.concat(sweeps).to_csv(ANALYSIS_DIR / "dev" / f"ri_dev{VERSION}_threshold_sweep.csv", index=False)
pd.DataFrame(best).to_csv(ANALYSIS_DIR / "dev" / f"ri_dev{VERSION}_best_thresholds.csv", index=False)
display(pd.DataFrame(quality)); display(pd.DataFrame(best))

## 6. Freeze the thresholds

From here on, the thresholds are read from `ri_dev<VERSION>_best_thresholds.csv` and never changed. Nothing below looks at the development set again.

In [ ]:
FROZEN = pd.read_csv(ANALYSIS_DIR / "dev" / f"ri_dev{VERSION}_best_thresholds.csv").set_index("model")["best threshold (PSS)"].to_dict()
FROZEN

## 7. Test set: scores with 95% confidence intervals

- **Bootstrap:** the 2,000 resamples redraw whole storms, separately among RI and non-RI storms. They are the same for every model, so the comparisons in section 8 are paired.
- **Population-weighted BSS:** the test set has 50 RI and 50 non-RI storms, but there are more non-RI storms in reality. This version weights each sampled storm by how many eligible storms of its group it stands for: storms in the test seasons with at least 2 observations.

In [ ]:
y = test["ri_tplus24"].values.astype(int)
draws = list(storm_bootstrap(test, np.random.default_rng(TEST_SEED), TEST_BOOTSTRAP))


def ci(fn, p):
    return np.percentile([fn(y[d], p[d]) for d in draws], [2.5, 97.5])


def fmt(value, lo_hi):
    return f"{value:.3f} [{lo_hi[0]:.3f}, {lo_hi[1]:.3f}]"


data = pd.read_csv(DATA_CSV, keep_default_na=False, na_values=[""], dtype={"sid": str})
storms = data.groupby("sid").agg(season=("season", "first"), n=("sid", "size"), ri=("ri_tplus24", "max"))
seasons = sorted(test["season"].unique())
pool = storms[storms["season"].between(seasons[0], seasons[-1]) & (storms["n"] >= 2)]
n_ri_pool, n_no_pool = int(pool["ri"].sum()), int((pool["ri"] == 0).sum())
n_ri_test = test.loc[test["storm_group"] == "ri", "sid"].nunique(); n_no_test = test.loc[test["storm_group"] == "no_ri", "sid"].nunique()
weights = np.where(test["storm_group"] == "ri", n_ri_pool / n_ri_test, n_no_pool / n_no_test)
print(f"eligible test-season storms: {n_ri_pool} with RI, {n_no_pool} without")

free, frozen = [], []
for m in models:
    p = test_p[m]
    assert not np.isnan(p).any(), f"{m}: unparseable test answers; decide how to score them before running this cell"
    free.append({"model": m, "ROC-AUC": fmt(roc_auc(y, p), ci(roc_auc, p)),
                 f"PR-AUC (random = {y.mean():.3f})": fmt(pr_auc(y, p), ci(pr_auc, p)),
                 "BSS vs dev climatology": fmt(bss(y, p), ci(bss, p)), "BSS, population-weighted": f"{bss(y, p, weights):.3f}",
                 "mean p / RI rate": f"{p.mean():.3f} / {y.mean():.3f}"})
    t = FROZEN[m]; s = at_threshold(y, p, t)
    frozen.append({"model": m, "frozen threshold": t,
                   **{k: fmt(s[k], ci(lambda yy, pp, k=k: at_threshold(yy, pp, t)[k], p)) for k in ["PSS", "POD", "FAR", "CSI"]}})

pd.DataFrame(free).to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_threshold_free.csv", index=False)
pd.DataFrame(frozen).to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_frozen_thresholds.csv", index=False)
display(pd.DataFrame(free)); display(pd.DataFrame(frozen))

## 8. Paired comparisons with the best model

- **What is compared:** for each other model, the difference in PR-AUC, BSS and ROC-AUC between `BEST_MODEL` and that model.
- **Confidence interval:** the 95% interval comes from the same 2,000 storm resamples.
- **p-value:** two-sided, from the share of resamples where the difference changes sign. A value of 0 means p < 1/2,000.

In [ ]:
out = []
if BEST_MODEL not in models:
    print(f"{BEST_MODEL} has no results for this version; no paired comparisons")
for m in (models if BEST_MODEL in models else []):
    if m == BEST_MODEL:
        continue
    for name, fn in [("PR-AUC", pr_auc), ("BSS", bss), ("ROC-AUC", roc_auc)]:
        diffs = np.array([fn(y[d], test_p[BEST_MODEL][d]) - fn(y[d], test_p[m][d]) for d in draws])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        out.append({"comparison": f"{BEST_MODEL} minus {m}", "metric": name,
                    "difference": round(fn(y, test_p[BEST_MODEL]) - fn(y, test_p[m]), 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
                    "p (bootstrap)": round(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())), 3)})
if out:
    pd.DataFrame(out).to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_paired_comparisons.csv", index=False)
    display(pd.DataFrame(out))

## 8b. Paired comparisons with the persistence baseline

- **What is compared:** each model minus `persistence` (notebook 11), in PR-AUC, BSS and ROC-AUC, on the same 2,000 storm resamples as section 7.
- **Reading it:** a model is significantly better than persistence when it wins in more than 97.5% of the resamples.
- Skipped if the persistence results are not present.

In [ ]:
REFERENCE = "persistence"
out = []
for m in (models if REFERENCE in models else []):
    if m == REFERENCE:
        continue
    for name, fn in [("PR-AUC", pr_auc), ("BSS", bss), ("ROC-AUC", roc_auc)]:
        diffs = np.array([fn(y[d], test_p[m][d]) - fn(y[d], test_p[REFERENCE][d]) for d in draws])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        out.append({"comparison": f"{m} minus {REFERENCE}", "metric": name,
                    "difference": round(fn(y, test_p[m]) - fn(y, test_p[REFERENCE]), 3),
                    "95% CI": f"[{lo:.3f}, {hi:.3f}]", "share of resamples won": round(float(np.mean(diffs > 0)), 3)})
vs_persistence = pd.DataFrame(out)
if len(vs_persistence):
    vs_persistence.to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_paired_vs_persistence.csv", index=False)
vs_persistence

## 9. Scores by phase, and calibration

**Phase**, from `usa_wind_delta_tminus24h`, the wind change over the past 24 h:

| Phase | Rule |
|---|---|
| RI already underway | change of 30 kt or more |
| quiet (onset) | change of less than 30 kt |
| no 24 h history | the change is missing |

Each cell is PR-AUC / ROC-AUC.

**Calibration** compares, within each band of predicted probability, the mean prediction with the observed RI rate.

In [ ]:
deltas = data[["sid", "iso_time", "usa_wind_delta_tminus24h"]]
phase_of = test[["sid", "iso_time"]].merge(deltas, on=["sid", "iso_time"], how="left", validate="one_to_one")["usa_wind_delta_tminus24h"]
test["phase"] = np.where(phase_of.isna(), "no 24 h history", np.where(phase_of >= 30, "RI already underway", "quiet (onset)"))

phases = []
for phase, g in test.groupby("phase"):
    k = g.index.values
    row = {"phase": phase, "cases (RI)": f"{len(k)} ({int(y[k].sum())})"}
    for m in models:
        row[f"{m} PR-AUC / ROC-AUC"] = f"{pr_auc(y[k], test_p[m][k]):.3f} / {roc_auc(y[k], test_p[m][k]):.3f}" if 0 < y[k].sum() < len(k) else "n/a"
    phases.append(row)
pd.DataFrame(phases).to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_by_phase.csv", index=False)
display(pd.DataFrame(phases))

cal = []
for m in models:
    bins = pd.cut(test_p[m], CALIBRATION_BINS, right=False)
    g = pd.DataFrame({"b": bins, "p": test_p[m], "y": y}).groupby("b", observed=True).agg(n=("y", "size"), mean_p=("p", "mean"), obs=("y", "mean"))
    cal.append(g.assign(model=m).reset_index())
cal = pd.concat(cal)
cal["bin"] = cal["b"].astype(str)
cal = cal.drop(columns="b")
cal.to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_calibration.csv", index=False)
display(cal.pivot_table(index="bin", columns="model", values=["n", "mean_p", "obs"], aggfunc="first").round(3))

### 9b. XGBoost against the best model, within each phase

The same paired storm bootstrap as section 8, restricted to the cases of one phase in every resample. A resample that has no RI case (or only RI cases) in a phase is skipped for that phase.

In [ ]:
TRAINED = "xgboost"
out = []
for phase in (sorted(test["phase"].unique()) if {TRAINED, BEST_MODEL} <= set(models) else []):
    in_phase = (test["phase"] == phase).values
    k = in_phase
    for name, fn in [("PR-AUC", pr_auc), ("ROC-AUC", roc_auc)]:
        diffs = []
        for d in draws:
            dd = d[in_phase[d]]
            if 0 < y[dd].sum() < len(dd):
                diffs.append(fn(y[dd], test_p[TRAINED][dd]) - fn(y[dd], test_p[BEST_MODEL][dd]))
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        out.append({"phase": phase, "metric": name, f"{TRAINED}": round(fn(y[k], test_p[TRAINED][k]), 3),
                    f"{BEST_MODEL}": round(fn(y[k], test_p[BEST_MODEL][k]), 3),
                    "difference": round(fn(y[k], test_p[TRAINED][k]) - fn(y[k], test_p[BEST_MODEL][k]), 3),
                    "95% CI": f"[{lo:.3f}, {hi:.3f}]", "share of resamples won": round(float(np.mean(np.array(diffs) > 0)), 3),
                    "resamples used": len(diffs)})
by_phase_paired = pd.DataFrame(out)
if len(by_phase_paired):
    by_phase_paired.to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_by_phase_paired.csv", index=False)
by_phase_paired

## 10. Robustness: gaps in the history

Notebook 01 keeps an observation only if its 24 h label exists. A storm's history can therefore skip a 6-hour slot when the storm was over land, no longer tropical, or missing 24 h after that slot. When the skipped slot lies within the last 24 h before t, the gap can hint at landfall or decay after t.

This table scores each model on all test cases and again without the cases that have such a gap. Similar numbers mean the results don't depend on the hint.

In [ ]:
t_case = pd.to_datetime(test["iso_time"])
times = t_case.groupby(test["sid"]).apply(set); first = t_case.groupby(test["sid"]).min()
gap = np.array([any(first[s] <= t - pd.Timedelta(hours=h) and t - pd.Timedelta(hours=h) not in times[s] for h in (6, 12, 18))
                for s, t in zip(test["sid"], t_case)])
print(f"test cases with a gap in their last 24 h: {gap.sum()} (RI rate {y[gap].mean():.3f}, against {y[~gap].mean():.3f} without)")

rows = []
for m in models:
    for label, k in [(f"all ({len(y)})", np.ones(len(y), bool)), (f"no gap ({int((~gap).sum())})", ~gap)]:
        rows.append({"model": m, "cases": label, "ROC-AUC": round(roc_auc(y[k], test_p[m][k]), 3),
                     "PR-AUC": round(pr_auc(y[k], test_p[m][k]), 3), "BSS": round(bss(y[k], test_p[m][k]), 3)})
pd.DataFrame(rows).to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_gap_robustness.csv", index=False)
display(pd.DataFrame(rows))

## 11. Robustness: probabilities recalibrated on the development set

The XGBoost baseline's probabilities are calibrated (on its training years); the LLMs' are used as given. To check that BSS differences are not only a matter of calibration, every model's probabilities are recalibrated the same way: an isotonic map fitted on the model's development-set answers, then applied to its test answers. Recalibration keeps the order of the probabilities, but isotonic regression merges nearby probabilities into ties, which can lower PR-AUC (shown for reference). The number of interest is BSS: how much each model gains from calibration alone. The XGBoost baseline's PR-AUC already includes the same kind of isotonic step.

In [ ]:
from sklearn.isotonic import IsotonicRegression

rows = []
for m in models:
    ok = ~np.isnan(dev_p[m])
    iso = IsotonicRegression(out_of_bounds="clip").fit(dev_p[m][ok], y_dev[ok])
    p_cal = iso.predict(test_p[m])
    rows.append({"model": m, "BSS as given": round(bss(y, test_p[m]), 3), "BSS recalibrated on dev": round(bss(y, p_cal), 3),
                 "PR-AUC as given": round(pr_auc(y, test_p[m]), 3), "PR-AUC recalibrated": round(pr_auc(y, p_cal), 3)})
pd.DataFrame(rows).to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_dev_recalibrated.csv", index=False)
display(pd.DataFrame(rows))

## 12. Image versus tabular input

For every language model: its score with images minus its score without, on the same test cases and the same 2,000 storm resamples (section 7). A change is significant when one sign wins in more than 97.5% of the resamples. Also the mean probability and the median number of input tokens per case in each mode, from the usage records in the results files. The table is the same whichever `VERSION` is set.

In [ ]:
import gzip

def test_cases(version):
    return ROOT / "model_inputs" / "ri_prediction" / "test" / f"test_2017-2025_50ri_50nori{version}.jsonl.gz"


def median_input_tokens(model, version):
    path = next(TEST_RESULTS.glob(f"*/{model}_{stem(test_cases(version))}.jsonl.gz"))
    with gzip.open(path, "rt") as f:
        usage = [json.loads(line)["response"].get("usage", {}) for line in f]
    tokens = [u.get("input_tokens", u.get("prompt_tokens")) for u in usage]
    return int(np.median([t for t in tokens if t is not None]))


t_tab, p_tab = load_split(test_cases("_text_only"), TEST_RESULTS)
t_img, p_img = load_split(test_cases("_text_image"), TEST_RESULTS)
assert (t_tab["case_id"].values == test["case_id"].values).all() and (t_img["case_id"].values == test["case_id"].values).all()
out = []
for m in [m for m in p_tab if m in p_img]:
    row = {"model": m}
    for name, fn in [("PR-AUC", pr_auc), ("BSS", bss), ("ROC-AUC", roc_auc)]:
        diffs = np.array([fn(y[d], p_img[m][d]) - fn(y[d], p_tab[m][d]) for d in draws])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        sig = max(np.mean(diffs > 0), np.mean(diffs < 0)) > 0.975
        row[f"{name} images minus tabular"] = f"{fn(y, p_img[m]) - fn(y, p_tab[m]):+.3f} [{lo:+.3f}, {hi:+.3f}]" + (" *" if sig else "")
    row["mean p tabular / images"] = f"{p_tab[m].mean():.3f} / {p_img[m].mean():.3f}"
    row["median input tokens tabular / images"] = f"{median_input_tokens(m, '_text_only')} / {median_input_tokens(m, '_text_image')}"
    out.append(row)
image_vs_tabular = pd.DataFrame(out)
image_vs_tabular.to_csv(ANALYSIS_DIR / "test" / "ri_test_image_vs_tabular.csv", index=False)
image_vs_tabular

### 12b. Images only

For the models that also answered the test cases with the picture and no tabular input (Haiku and Sonnet): their scores with 95% storm-bootstrap intervals, and paired differences with their own tabular and tabular + image runs and with persistence, on the same 2,000 resamples as section 7. `*` marks a difference won in more than 97.5% of resamples (either sign).

In [ ]:
t_only, p_only = load_split(test_cases("_image_only"), TEST_RESULTS)
assert (t_only["case_id"].values == test["case_id"].values).all()
out = []
for m in [m for m in p_only if m in p_tab and m in p_img]:
    row = {"model": m}
    for name, fn in [("PR-AUC", pr_auc), ("BSS", bss), ("ROC-AUC", roc_auc)]:
        row[f"{name} images only"] = fmt(fn(y, p_only[m]), ci(fn, p_only[m]))
        others = [("tabular", p_tab[m]), ("tabular + images", p_img[m])]
        if "persistence" in test_p:
            others.append(("persistence", test_p["persistence"]))
        for label, other in others:
            diffs = np.array([fn(y[d], p_only[m][d]) - fn(y[d], other[d]) for d in draws])
            lo, hi = np.percentile(diffs, [2.5, 97.5])
            sig = max(np.mean(diffs > 0), np.mean(diffs < 0)) > 0.975
            row[f"{name} images only minus {label}"] = f"{fn(y, p_only[m]) - fn(y, other):+.3f} [{lo:+.3f}, {hi:+.3f}]" + (" *" if sig else "")
    row["mean p images only"] = round(float(np.mean(p_only[m])), 3)
    out.append(row)
image_only = pd.DataFrame(out)
if len(image_only):
    image_only.to_csv(ANALYSIS_DIR / "test" / "ri_test_image_only_vs_other_inputs.csv", index=False)
image_only.T

## 13. Yes/no scores: XGBoost against the best model

PSS and CSI at the frozen thresholds (section 6), paired on the same storm resamples. Skipped if XGBoost has no results for this version.

In [ ]:
def yes_no(y_, p_, t):
    warn = p_ >= t
    hits, false_alarms = (warn & (y_ == 1)).sum(), (warn & (y_ == 0)).sum()
    misses = (~warn & (y_ == 1)).sum()
    pss = hits / (y_ == 1).sum() - false_alarms / (y_ == 0).sum()
    return pss, hits / (hits + false_alarms + misses)


out = []
if {"xgboost", BEST_MODEL} <= set(models):
    for k, name in enumerate(["PSS", "CSI"]):
        score = lambda m, d: yes_no(y[d], test_p[m][d], FROZEN[m])[k]
        everything = np.arange(len(y))
        diffs = np.array([score("xgboost", d) - score(BEST_MODEL, d) for d in draws])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        out.append({"comparison": f"xgboost minus {BEST_MODEL}", "metric": name,
                    "difference": round(score("xgboost", everything) - score(BEST_MODEL, everything), 3),
                    "95% CI": f"[{lo:.3f}, {hi:.3f}]", "share of resamples won": round(float(np.mean(diffs > 0)), 3)})
yes_no_paired = pd.DataFrame(out)
if len(yes_no_paired):
    yes_no_paired.to_csv(ANALYSIS_DIR / "test" / f"ri_test{VERSION}_yes_no_paired.csv", index=False)
yes_no_paired

## Notes

- **Order matters:** the thresholds in section 6 must come from the development set alone. Never edit `ri_dev<VERSION>_best_thresholds.csv` by hand after seeing test results.
- **Adding a model:** put its development and test results files in the two results folders, named like the others, and re-run the notebook. Every table is rebuilt.
- **Unparseable answers:** on the development set they are left out of that model's scores (and listed). On the test set the notebook stops, so you decide how to score them before any test number is produced.